In [6]:
%%configure -f
{
  "defaultLakehouse": {
    "name": "dataverse_esacontact_cds2_workspace_a94a4bf848e144bba608bb2eb51cbe",
    "id": "ee1e74a0-c7a7-423e-8d8d-d4df8faa1b17",
    "workspaceId": "6988ba37-18e0-4677-82f3-aab0aa8112cc"
  }
}

In [8]:
spark.sql("""
CREATE TABLE IF NOT EXISTS DateTable
USING DELTA
LOCATION 'abfss://6988ba37-18e0-4677-82f3-aab0aa8112cc@onelake.dfs.fabric.microsoft.com/ee1e74a0-c7a7-423e-8d8d-d4df8faa1b17/Tables/DateTable'
""")

spark.sql("DESCRIBE TABLE DateTable").show(truncate=False)

In [7]:
from pyspark.sql import functions as F

LAKEHOUSE_TABLES = "abfss://6988ba37-18e0-4677-82f3-aab0aa8112cc@onelake.dfs.fabric.microsoft.com/ee1e74a0-c7a7-423e-8d8d-d4df8faa1b17/Tables"

df = spark.sql("""
    SELECT explode(
        sequence(
            to_date('2015-01-01'),
            to_date('2030-12-31'),
            interval 1 day
        )
    ) AS Date
""")

df = (
    df.withColumn("Year", F.year("Date"))
      .withColumn("MonthNumber", F.month("Date"))
      .withColumn("MonthName", F.date_format("Date", "MMMM"))
      .withColumn("Quarter", F.concat(F.lit("Q"), F.quarter("Date")))
)

df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(f"{LAKEHOUSE_TABLES}/DateTable")

print("DateTable written:", df.count(), "rows")
df.show(5)